# Find and load the data

In [2]:
from pathlib import Path
import json
import time

import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# Use the same random seed for repeatable experiments
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Check that Kaggle assigned a GPU
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus if gpus else "No GPU found")

# Find the uploaded dataset file
files = list(Path("/kaggle/input").rglob("brainscan_data.npz"))

if not files:
    raise FileNotFoundError(
        "brainscan_data.npz was not found. Check that the Kaggle Dataset is attached."
    )

data_path = files[0]
print("Dataset:", data_path)

# Load the training and validation data.
# We intentionally do not load the test set during training.
with np.load(data_path, allow_pickle=False) as data:
    X_train = data["X_train"].astype("float32")
    y_train = data["y_train"].astype("float32")
    X_val = data["X_val"].astype("float32")
    y_val = data["y_val"].astype("float32")
    class_names = [str(name) for name in data["class_names"]]

num_classes = y_train.shape[1]

print("Classes:", class_names)
print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Validation images:", X_val.shape)
print("Validation labels:", y_val.shape)
print("Pixel range:", float(X_train.min()), "to", float(X_train.max()))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Dataset: /kaggle/input/datasets/hamzalafsioui/brainscan-data/brainscan_data.npz
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training images: (4915, 224, 224, 3)
Training labels: (4915, 4)
Validation images: (703, 224, 224, 3)
Validation labels: (703, 4)
Pixel range: 0.0 to 1.0


# Check the classes before continuing

In [3]:
if len(class_names) != num_classes:
    raise ValueError("The number of class names does not match the label columns.")

if num_classes != 4:
    raise ValueError(
        f"This version expects 2 classes, but found {class_names}. "
        "Fix the labels before training the binary classifier."
    )

if X_train.max() > 1.0:
    raise ValueError(
        "Images are not normalized to [0, 1]. Check the preprocessing notebook."
    )

# Calculate class weights

In [4]:
train_labels = np.argmax(y_train, axis=1)
class_ids = np.unique(train_labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=train_labels,
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(class_ids, weights)
}

print("Training images per class:",
      np.bincount(train_labels, minlength=num_classes))
print("Class weights:", class_weights)

Training images per class: [1135 1151 1400 1229]
Class weights: {0: 1.0825991189427313, 1: 1.0675499565595135, 2: 0.8776785714285714, 3: 0.9997965825874695}


# Build the CNN

In [5]:
def build_cnn(input_shape, num_classes, learning_rate=1e-4):
    model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=input_shape),

            tf.keras.layers.Conv2D(
                32, (3, 3), padding="same", activation="relu"
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),

            tf.keras.layers.Conv2D(
                64, (3, 3), padding="same", activation="relu"
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),

            tf.keras.layers.Conv2D(
                128, (3, 3), padding="same", activation="relu"
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),
            tf.keras.layers.Dropout(0.25),

            tf.keras.layers.Conv2D(
                128, (3, 3), padding="same", activation="relu"
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling2D((2, 2)),
            tf.keras.layers.Dropout(0.25),

            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(num_classes, activation="softmax"),
        ],
        name="BrainScan_CNN",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


model = build_cnn(
    input_shape=X_train.shape[1:],
    num_classes=num_classes,
)

model.summary()

I0000 00:00:1792146438.709518      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792146438.713032      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "BrainScan_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,148 (13.18 MB)

 Trainable params: 3,453,444 (13.17 MB)

 Non-trainable params: 704 (2.75 KB)

# Train and save the best model

In [6]:
output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)

# Native Keras format is recommended for saving Keras models.
model_path = output_dir / "best_brainscan.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

start_time = time.perf_counter()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)

training_seconds = time.perf_counter() - start_time

best_epoch = int(np.argmin(history.history["val_loss"])) + 1

print(f"Training time: {training_seconds / 60:.1f} minutes")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")
print("Best model saved to:", model_path)

Epoch 1/30
  2/154 ━━━━━━━━━━━━━━━━━━━━ 12s 80ms/step - accuracy: 0.2422 - loss: 4.0584 

I0000 00:00:1792146465.239627     169 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 0.5851 - loss: 1.3555
Epoch 1: val_loss improved from None to 6.92859, saving model to /kaggle/working/best_brainscan.keras

Epoch 1: finished saving model to /kaggle/working/best_brainscan.keras
154/154 ━━━━━━━━━━━━━━━━━━━━ 33s 136ms/step - accuracy: 0.6665 - loss: 0.9249 - val_accuracy: 0.2304 - val_loss: 6.9286 - learning_rate: 1.0000e-04
Epoch 2/30
153/154 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.7638 - loss: 0.5954
Epoch 2: val_loss improved from 6.92859 to 4.76329, saving model to /kaggle/working/best_brainscan.keras

Epoch 2: finished saving model to /kaggle/working/best_brainscan.keras
154/154 ━━━━━━━━━━━━━━━━━━━━ 10s 68ms/step - accuracy: 0.7758 - loss: 0.5655 - val_accuracy: 0.2660 - val_loss: 4.7633 - learning_rate: 1.0000e-04
Epoch 3/30
153/154 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.8229 - loss: 0.4631
Epoch 3: val_loss improved from 4.76329 to 1.35020, saving model to /kaggle/working/best_brainscan.kera

# Save the class names and training history

In [7]:
with open(output_dir / "class_names.json", "w", encoding="utf-8") as file:
    json.dump(class_names, file, ensure_ascii=False, indent=2)

history_to_save = {
    key: [float(value) for value in values]
    for key, values in history.history.items()
}

with open(output_dir / "training_history.json", "w", encoding="utf-8") as file:
    json.dump(history_to_save, file, indent=2)

print("Saved files:")
for path in output_dir.iterdir():
    print("-", path.name)

Saved files:
- training_history.json
- best_brainscan.keras
- .virtual_documents
- class_names.json


In [1]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2
